# S02 — QC, Cohort Availability, Derived Features, and Non-destructive Transformations

This notebook:
1) loads canonical data from S1 (`pcos_base.parquet` preferred, fallback to CSV),
2) performs non-destructive QC:
   - missingness
   - sanity-range flags
   - structural data-quality flags
3) computes derived features:
   - TG/HDL ratio
   - non-HDL cholesterol
4) applies non-destructive transformations:
   - log1p for config-driven skewed variables
   - optional winsorized copies (never overwrite raw variables)
5) exports:
   - processed dataset (`pcos_qc.parquet`, `pcos_qc.csv`)
   - QC tables
   - cohort availability tables for downstream STROBE/flow reporting
   - compact QC summary JSON

Important:
- S2 does NOT exclude rows from the analytical cohort.
- S2 does NOT perform endpoint-specific trimming.
- All exclusions, if any, should be defined explicitly downstream.

## Imports

In [ ]:
import json
import logging
from pathlib import Path
from typing import Dict, Any, Optional, List, Tuple

import numpy as np
import pandas as pd

## Config

In [ ]:
def load_json(path: Path) -> dict:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def resolve_config_path() -> Path:
    candidates = [
        Path("/content/reports/config_snapshot.json"),
        Path("/mnt/data/config_snapshot.json"),
        Path("/content/config.json"),
        Path("/mnt/data/config.json"),
    ]
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError(
        "No config found. Expected config_snapshot.json or config.json in /content or /mnt/data."
    )

CONFIG_PATH = resolve_config_path()
CFG = load_json(CONFIG_PATH)

print("Loaded config:", str(CONFIG_PATH))

Loaded config: /content/config.json


## Directories and logging

In [ ]:
def ensure_dirs(cfg: dict) -> None:
    path_keys = [
        "output_dir",
        "intermediate_dir",
        "figures_dir",
        "tables_dir",
        "models_dir",
        "reports_dir",
        "qc_dir",
        "supplementary_dir",
    ]
    for key in path_keys:
        if key in cfg.get("paths", {}):
            Path(cfg["paths"][key]).mkdir(parents=True, exist_ok=True)

def setup_logging(cfg: dict) -> None:
    if not cfg.get("logging", {}).get("enabled", False):
        return

    root_logger = logging.getLogger()
    if root_logger.handlers:
        root_logger.setLevel(logging.INFO)
        return

    log_file = Path(cfg["logging"]["log_file"])
    log_file.parent.mkdir(parents=True, exist_ok=True)

    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s | %(levelname)s | %(message)s",
        handlers=[
            logging.FileHandler(log_file, encoding="utf-8"),
            logging.StreamHandler(),
        ],
    )
    logging.info("Logging initialized in S2.")
    logging.info("Config loaded from: %s", str(CONFIG_PATH))

ensure_dirs(CFG)
setup_logging(CFG)

paths = CFG["paths"]
intermediate_dir = Path(paths["intermediate_dir"])
tables_dir = Path(paths["tables_dir"])
qc_dir = Path(paths["qc_dir"])
reports_dir = Path(paths["reports_dir"])

## Load data

In [ ]:
def resolve_base_data_paths(cfg: dict) -> List[Path]:
    candidates = [
        Path(cfg["paths"]["intermediate_dir"]) / "pcos_base.parquet",
        Path(cfg["paths"]["intermediate_dir"]) / "pcos_base.csv",
        Path("/content/pcos_base.parquet"),
        Path("/content/pcos_base.csv"),
        Path("/mnt/data/pcos_base.parquet"),
        Path("/mnt/data/pcos_base.csv"),
    ]
    return candidates

def load_base_dataset(cfg: dict) -> Tuple[pd.DataFrame, str]:
    candidates = resolve_base_data_paths(cfg)

    parquet_candidates = [p for p in candidates if p.suffix == ".parquet" and p.exists()]
    csv_candidates = [p for p in candidates if p.suffix == ".csv" and p.exists()]

    for p in parquet_candidates:
        try:
            df = pd.read_parquet(p)
            return df, str(p)
        except Exception as e:
            logging.warning("Failed to read parquet %s: %r", str(p), e)

    for p in csv_candidates:
        try:
            df = pd.read_csv(p)
            return df, str(p)
        except Exception as e:
            logging.warning("Failed to read csv %s: %r", str(p), e)

    raise FileNotFoundError(
        "Could not locate pcos_base.parquet or pcos_base.csv in configured/intermediate locations."
    )

df, source_used = load_base_dataset(CFG)

print("Loaded:", source_used)
print("Shape:", df.shape)
df.head(3)

Loaded: /content/pcos_base.parquet
Shape: (1300, 25)


,id,age,anti_tpo,anti_tg,tsh,ft4,ft3,tg,hdl,tc,...,shbg,dheas,andro,amh,lh,fsh,ins0,ins120,insulin,lt4_use
0,7611,25.0,13.8,NaN,0.969,1.20,NaN,116.0,56.6,188.0,...,NaN,379.0,2.91,NaN,33.60,10.10,NaN,NaN,11.9,NaN
1,8133,25.0,12.6,NaN,2.050,1.18,NaN,144.0,41.9,196.0,...,NaN,340.0,0.72,NaN,3.08,5.45,NaN,NaN,10.2,NaN
2,11028,25.0,150.0,NaN,2.500,1.29,NaN,35.8,62.7,133.0,...,NaN,462.0,1.72,NaN,5.65,7.03,NaN,NaN,20.7,NaN


## Auxiliary functions

In [ ]:
def missingness_report(df: pd.DataFrame) -> pd.DataFrame:
    rep = pd.DataFrame({
        "variable": df.columns,
        "n_missing": df.isna().sum().values,
        "pct_missing": (df.isna().mean() * 100).round(2).values,
        "dtype": df.dtypes.astype(str).values,
        "n_non_missing": df.notna().sum().values,
    }).sort_values(["pct_missing", "n_missing", "variable"], ascending=[False, False, True])
    return rep.reset_index(drop=True)

def coerce_numeric(series: pd.Series) -> pd.Series:
    return pd.to_numeric(series, errors="coerce")

def winsorize_series(x: pd.Series, q_low: float, q_high: float) -> pd.Series:
    x = coerce_numeric(x)
    if x.notna().sum() == 0:
        return x
    lo = x.quantile(q_low)
    hi = x.quantile(q_high)
    return x.clip(lower=lo, upper=hi)

def safe_log1p(x: pd.Series) -> pd.Series:
    x = coerce_numeric(x)
    x = x.where(x >= 0, np.nan)
    return np.log1p(x)

def safe_divide(a: pd.Series, b: pd.Series) -> pd.Series:
    a = coerce_numeric(a)
    b = coerce_numeric(b)
    b = b.where(b > 0, np.nan)
    return a / b

def add_outlier_flag(df: pd.DataFrame, col: str, low: Optional[float], high: Optional[float]) -> pd.DataFrame:
    if col not in df.columns:
        return df

    x = coerce_numeric(df[col])
    flag = pd.Series(False, index=df.index)

    if low is not None:
        flag = flag | (x < low)
    if high is not None:
        flag = flag | (x > high)

    df[f"out_{col}"] = flag.fillna(False)
    return df

def add_structural_flag(df: pd.DataFrame, col: str, rule_name: str, mask: pd.Series) -> pd.DataFrame:
    if col not in df.columns:
        return df
    df[f"flag_{col}_{rule_name}"] = mask.fillna(False)
    return df

def summarize_numeric(series: pd.Series) -> Dict[str, Any]:
    x = coerce_numeric(series)
    if x.notna().sum() == 0:
        return {
            "n_non_missing": 0,
            "min": np.nan,
            "p1": np.nan,
            "median": np.nan,
            "p99": np.nan,
            "max": np.nan,
        }
    return {
        "n_non_missing": int(x.notna().sum()),
        "min": float(x.min()),
        "p1": float(x.quantile(0.01)),
        "median": float(x.median()),
        "p99": float(x.quantile(0.99)),
        "max": float(x.max()),
    }

## Sanity bounds

In [ ]:
SANITY_BOUNDS = {
    "age": (10, 60),

    # Thyroid
    "tsh": (0.01, 100.0),
    "anti_tpo": (0.0, 5000.0),
    "anti_tg": (0.0, 5000.0),
    "ft4": (0.0, 100.0),
    "ft3": (0.0, 100.0),

    # Lipids (mg/dL)
    "tg": (0.0, 3000.0),
    "hdl": (0.0, 200.0),
    "tc": (0.0, 1000.0),
    "ldl": (0.0, 1000.0),

    # Glucose (mg/dL)
    "glu0": (0.0, 600.0),
    "glu120": (0.0, 600.0),

    # Androgens / hormones
    "tt": (0.0, 1000.0),
    "ft": (0.0, 1000.0),
    "shbg": (0.0, 10000.0),
    "dheas": (0.0, 10000.0),
    "andro": (0.0, 10000.0),
    "amh": (0.0, 1000.0),
    "lh": (0.0, 500.0),
    "fsh": (0.0, 500.0),
}

## Main QC: flags but no observation removal

In [ ]:
df_qc = df.copy()


for col, (lo, hi) in SANITY_BOUNDS.items():
    df_qc = add_outlier_flag(df_qc, col, lo, hi)


if "hdl" in df_qc.columns:
    df_qc = add_structural_flag(df_qc, "hdl", "nonpositive", coerce_numeric(df_qc["hdl"]) <= 0)

if "tg" in df_qc.columns:
    df_qc = add_structural_flag(df_qc, "tg", "negative", coerce_numeric(df_qc["tg"]) < 0)

if "tc" in df_qc.columns:
    df_qc = add_structural_flag(df_qc, "tc", "negative", coerce_numeric(df_qc["tc"]) < 0)

if "glu0" in df_qc.columns:
    df_qc = add_structural_flag(df_qc, "glu0", "negative", coerce_numeric(df_qc["glu0"]) < 0)

if "glu120" in df_qc.columns:
    df_qc = add_structural_flag(df_qc, "glu120", "negative", coerce_numeric(df_qc["glu120"]) < 0)

outlier_cols = [c for c in df_qc.columns if c.startswith("out_")]
flag_cols = [c for c in df_qc.columns if c.startswith("flag_")]

print("Outlier flag columns:", len(outlier_cols))
print("Structural flag columns:", len(flag_cols))

if outlier_cols:
    display(df_qc[outlier_cols].sum().sort_values(ascending=False).head(15))
if flag_cols:
    display(df_qc[flag_cols].sum().sort_values(ascending=False).head(15))

Outlier flag columns: 20
Structural flag columns: 5


,0
out_tsh,1
out_age,0
out_anti_tpo,0
out_anti_tg,0
out_ft4,0
out_ft3,0
out_tg,0
out_hdl,0
out_tc,0
out_ldl,0


,0
flag_hdl_nonpositive,0
flag_tg_negative,0
flag_tc_negative,0
flag_glu0_negative,0
flag_glu120_negative,0


## Derived features

In [ ]:
# TG/HDL ratio
if {"tg", "hdl"}.issubset(df_qc.columns):
    df_qc["tg_hdl_ratio"] = safe_divide(df_qc["tg"], df_qc["hdl"])
else:
    df_qc["tg_hdl_ratio"] = np.nan

# non-HDL = TC - HDL
if {"tc", "hdl"}.issubset(df_qc.columns):
    df_qc["non_hdl"] = coerce_numeric(df_qc["tc"]) - coerce_numeric(df_qc["hdl"])
else:
    df_qc["non_hdl"] = np.nan

# QC flags for derived features
df_qc = add_outlier_flag(df_qc, "tg_hdl_ratio", 0.0, 100.0)
df_qc = add_outlier_flag(df_qc, "non_hdl", -50.0, 1000.0)

df_qc = add_structural_flag(
    df_qc,
    "non_hdl",
    "negative",
    coerce_numeric(df_qc["non_hdl"]) < 0
)

preview_cols = [c for c in ["tg", "hdl", "tg_hdl_ratio", "tc", "non_hdl"] if c in df_qc.columns]
df_qc[preview_cols].head(5)

,tg,hdl,tg_hdl_ratio,tc,non_hdl
0,116.0,56.6,2.049470,188.0,131.4
1,144.0,41.9,3.436754,196.0,154.1
2,35.8,62.7,0.570973,133.0,70.3
3,73.3,58.1,1.261618,147.0,88.9
4,149.0,39.0,3.820513,150.0,111.0


## Transformations

In [ ]:
log_candidates = CFG.get("transforms", {}).get("log1p_candidates", [])

for col in log_candidates:
    if col in df_qc.columns:
        df_qc[f"log1p_{col}"] = safe_log1p(df_qc[col])

winsor_low = float(CFG.get("transforms", {}).get("winsor_q_low", 0.01))
winsor_high = float(CFG.get("transforms", {}).get("winsor_q_high", 0.99))


winsor_enabled = bool(
    CFG.get("transforms", {}).get("winsorize", {}).get("enabled", False)
)

winsor_targets = list(dict.fromkeys(log_candidates + ["tg_hdl_ratio", "non_hdl"]))

if winsor_enabled:
    for col in winsor_targets:
        if col in df_qc.columns:
            df_qc[f"win_{col}"] = winsorize_series(df_qc[col], winsor_low, winsor_high)

print("Added log1p columns:", [c for c in df_qc.columns if c.startswith("log1p_")])
print("Added winsorized columns:", [c for c in df_qc.columns if c.startswith("win_")])
print("Winsorization enabled:", winsor_enabled)

Added log1p columns: ['log1p_anti_tpo', 'log1p_tg', 'log1p_ins0', 'log1p_ins120']
Added winsorized columns: []
Winsorization enabled: False


## Missingness report

In [ ]:
miss = missingness_report(df_qc)
out_miss = qc_dir / "qc_missingness_report.csv"
miss.to_csv(out_miss, index=False)

print("Saved missingness report:", out_miss)
miss.head(20)

Saved missingness report: /content/outputs/qc/qc_missingness_report.csv


,variable,n_missing,pct_missing,dtype,n_non_missing
0,ins0,1300,100.00,float64,0
1,ins120,1300,100.00,float64,0
2,log1p_ins0,1300,100.00,float64,0
3,log1p_ins120,1300,100.00,float64,0
4,lt4_use,1300,100.00,float64,0
5,shbg,1299,99.92,float64,1
6,ft3,1294,99.54,float64,6
7,amh,1153,88.69,float64,147
8,anti_tg,1043,80.23,float64,257
9,andro,609,46.85,float64,691


## Summary outlier/flag report

In [ ]:
summary_rows = []

summary_vars = [c for c in SANITY_BOUNDS.keys() if c in df_qc.columns]
summary_vars += [c for c in ["tg_hdl_ratio", "non_hdl"] if c in df_qc.columns]

for c in summary_vars:
    stats = summarize_numeric(df_qc[c])

    outflag = df_qc.get(f"out_{c}", pd.Series(False, index=df_qc.index))
    structural_cols = [col for col in df_qc.columns if col.startswith(f"flag_{c}_")]

    n_structural_flags = 0
    if structural_cols:
        n_structural_flags = int(df_qc[structural_cols].any(axis=1).sum())

    lo, hi = SANITY_BOUNDS.get(c, (np.nan, np.nan))

    summary_rows.append({
        "variable": c,
        **stats,
        "sanity_low": lo,
        "sanity_high": hi,
        "n_flagged_outliers": int(outflag.sum()) if isinstance(outflag, pd.Series) else np.nan,
        "pct_flagged_outliers": float((outflag.mean() * 100).round(2)) if isinstance(outflag, pd.Series) else np.nan,
        "n_structural_flags": n_structural_flags,
        "pct_structural_flags": round((n_structural_flags / len(df_qc)) * 100, 2) if len(df_qc) else np.nan,
    })

out_summary = pd.DataFrame(summary_rows).sort_values(
    ["pct_flagged_outliers", "n_structural_flags", "n_non_missing"],
    ascending=[False, False, False]
)

out_summary_path = qc_dir / "qc_outlier_summary.csv"
out_summary.to_csv(out_summary_path, index=False)

print("Saved outlier/flag summary:", out_summary_path)
out_summary.head(20)

Saved outlier/flag summary: /content/outputs/qc/qc_outlier_summary.csv


,variable,n_non_missing,min,p1,median,p99,max,sanity_low,sanity_high,n_flagged_outliers,pct_flagged_outliers,n_structural_flags,pct_structural_flags
1,tsh,1182,0.005000,0.422050,1.820000,5.594600,15.000000,0.01,100.0,1,0.08,0,0.0
0,age,1300,16.000000,17.000000,21.000000,24.000000,25.000000,10.00,60.0,0,0.00,0,0.0
15,dheas,1185,30.900000,106.680000,321.000000,754.800000,1185.000000,0.00,10000.0,0,0.00,0,0.0
6,tg,1183,24.000000,34.982000,77.200000,239.340000,370.000000,0.00,3000.0,0,0.00,0,0.0
7,hdl,1183,22.500000,30.700000,56.900000,93.590000,174.000000,0.00,200.0,0,0.00,0,0.0
8,tc,1183,94.000000,107.820000,165.000000,248.180000,345.000000,0.00,1000.0,0,0.00,0,0.0
9,ldl,1183,20.300000,42.156000,88.580000,160.162000,239.000000,0.00,1000.0,0,0.00,0,0.0
20,tg_hdl_ratio,1183,0.351322,0.463298,1.313993,6.468349,13.214286,NaN,NaN,0,0.00,0,0.0
21,non_hdl,1183,27.000000,51.946000,104.700000,189.000000,259.000000,NaN,NaN,0,0.00,0,0.0
18,lh,1182,0.100000,0.638100,7.075000,29.819000,60.610000,0.00,500.0,0,0.00,0,0.0


## Availability / cohort flow support

In [ ]:
def row_complete_for(df: pd.DataFrame, cols: List[str]) -> pd.Series:
    available_cols = [c for c in cols if c in df.columns]
    if not available_cols:
        return pd.Series(False, index=df.index)
    return df[available_cols].notna().all(axis=1)

eligibility_cfg = CFG.get("eligibility", {})

required_primary = eligibility_cfg.get("required_for_primary", [])
required_non_hdl = eligibility_cfg.get("required_for_secondary_non_hdl", [])
required_ogtt120 = eligibility_cfg.get("required_for_secondary_ogtt120", [])

availability = pd.DataFrame({
    "has_age": df_qc["age"].notna() if "age" in df_qc.columns else False,
    "has_anti_tpo": df_qc["anti_tpo"].notna() if "anti_tpo" in df_qc.columns else False,
    "has_tg": df_qc["tg"].notna() if "tg" in df_qc.columns else False,
    "has_hdl": df_qc["hdl"].notna() if "hdl" in df_qc.columns else False,
    "has_tc": df_qc["tc"].notna() if "tc" in df_qc.columns else False,
    "has_glu120": df_qc["glu120"].notna() if "glu120" in df_qc.columns else False,
    "eligible_primary_minimal": row_complete_for(df_qc, required_primary),
    "eligible_non_hdl_minimal": row_complete_for(df_qc, required_non_hdl),
    "eligible_ogtt120_minimal": row_complete_for(df_qc, required_ogtt120),
})

availability["id"] = df_qc["id"] if "id" in df_qc.columns else np.arange(len(df_qc))

availability_path = qc_dir / "qc_cohort_availability.csv"
availability.to_csv(availability_path, index=False)

availability_summary = pd.DataFrame([
    {"stage": "n_total_rows", "n": int(len(df_qc))},
    {"stage": "has_age", "n": int(availability["has_age"].sum())},
    {"stage": "has_anti_tpo", "n": int(availability["has_anti_tpo"].sum())},
    {"stage": "has_tg", "n": int(availability["has_tg"].sum())},
    {"stage": "has_hdl", "n": int(availability["has_hdl"].sum())},
    {"stage": "has_tc", "n": int(availability["has_tc"].sum())},
    {"stage": "has_glu120", "n": int(availability["has_glu120"].sum())},
    {"stage": "eligible_primary_minimal", "n": int(availability["eligible_primary_minimal"].sum())},
    {"stage": "eligible_non_hdl_minimal", "n": int(availability["eligible_non_hdl_minimal"].sum())},
    {"stage": "eligible_ogtt120_minimal", "n": int(availability["eligible_ogtt120_minimal"].sum())},
])

availability_summary_path = qc_dir / "qc_cohort_availability_summary.csv"
availability_summary.to_csv(availability_summary_path, index=False)

print("Saved:", availability_path)
print("Saved:", availability_summary_path)
availability_summary

Saved: /content/outputs/qc/qc_cohort_availability.csv
Saved: /content/outputs/qc/qc_cohort_availability_summary.csv


,stage,n
0,n_total_rows,1300
1,has_age,1300
2,has_anti_tpo,1055
3,has_tg,1183
4,has_hdl,1183
5,has_tc,1183
6,has_glu120,1162
7,eligible_primary_minimal,1053
8,eligible_non_hdl_minimal,1053
9,eligible_ogtt120_minimal,1035


## Cohort age summary

In [ ]:
age_summary = {}

if "age" in df_qc.columns:
    age = coerce_numeric(df_qc["age"])
    age_summary = {
        "n_non_missing_age": int(age.notna().sum()),
        "age_min": float(age.min()) if age.notna().any() else np.nan,
        "age_p25": float(age.quantile(0.25)) if age.notna().any() else np.nan,
        "age_median": float(age.median()) if age.notna().any() else np.nan,
        "age_p75": float(age.quantile(0.75)) if age.notna().any() else np.nan,
        "age_max": float(age.max()) if age.notna().any() else np.nan,
        "age_mean": float(age.mean()) if age.notna().any() else np.nan,
        "age_sd": float(age.std()) if age.notna().any() else np.nan,
    }

age_summary_df = pd.DataFrame([age_summary])
age_summary_path = qc_dir / "qc_age_summary.csv"
age_summary_df.to_csv(age_summary_path, index=False)

print("Saved:", age_summary_path)
age_summary_df

Saved: /content/outputs/qc/qc_age_summary.csv


,n_non_missing_age,age_min,age_p25,age_median,age_p75,age_max,age_mean,age_sd
0,1300,16.0,19.0,21.0,22.0,25.0,20.923077,1.948985


## QC compact JSON

In [ ]:
qc_summary = {
    "source_used": source_used,
    "n_rows": int(len(df_qc)),
    "n_columns": int(df_qc.shape[1]),
    "config_path": str(CONFIG_PATH),
    "winsorization_enabled": winsor_enabled,
    "log1p_candidates_from_config": log_candidates,
    "derived_features_created": [c for c in ["tg_hdl_ratio", "non_hdl"] if c in df_qc.columns],
    "availability_summary": availability_summary.to_dict(orient="records"),
    "age_summary": age_summary,
    "notable_unmapped_or_missing_analysis_fields": {
        "lt4_use_present": bool("lt4_use" in df_qc.columns),
        "ins0_present": bool("ins0" in df_qc.columns),
        "ins120_present": bool("ins120" in df_qc.columns),
    },
}

qc_summary_path = reports_dir / "qc_summary.json"
with open(qc_summary_path, "w", encoding="utf-8") as f:
    json.dump(qc_summary, f, indent=2, ensure_ascii=False)

print("Saved:", qc_summary_path)
qc_summary

Saved: /content/reports/qc_summary.json


{'source_used': '/content/pcos_base.parquet',
 'n_rows': 1300,
 'n_columns': 59,
 'config_path': '/content/config.json',
 'winsorization_enabled': False,
 'log1p_candidates_from_config': ['anti_tpo', 'tg', 'ins0', 'ins120'],
 'derived_features_created': ['tg_hdl_ratio', 'non_hdl'],
 'availability_summary': [{'stage': 'n_total_rows', 'n': 1300},
  {'stage': 'has_age', 'n': 1300},
  {'stage': 'has_anti_tpo', 'n': 1055},
  {'stage': 'has_tg', 'n': 1183},
  {'stage': 'has_hdl', 'n': 1183},
  {'stage': 'has_tc', 'n': 1183},
  {'stage': 'has_glu120', 'n': 1162},
  {'stage': 'eligible_primary_minimal', 'n': 1053},
  {'stage': 'eligible_non_hdl_minimal', 'n': 1053},
  {'stage': 'eligible_ogtt120_minimal', 'n': 1035}],
 'age_summary': {'n_non_missing_age': 1300,
  'age_min': 16.0,
  'age_p25': 19.0,
  'age_median': 21.0,
  'age_p75': 22.0,
  'age_max': 25.0,
  'age_mean': 20.923076923076923,
  'age_sd': 1.9489851867388472},
 'notable_unmapped_or_missing_analysis_fields': {'lt4_use_present': Tru

## Saving the final dataset

In [ ]:
out_parquet = intermediate_dir / "pcos_qc.parquet"
out_csv = intermediate_dir / "pcos_qc.csv"

try:
    df_qc.to_parquet(out_parquet, index=False)
    print("Saved:", out_parquet)
except Exception as e:
    print("Parquet save failed. Reason:", repr(e))
    logging.warning("Parquet save failed: %r", e)

df_qc.to_csv(out_csv, index=False)
print("Saved:", out_csv)

print("Final shape:", df_qc.shape)

Saved: /content/outputs/intermediate/pcos_qc.parquet
Saved: /content/outputs/intermediate/pcos_qc.csv
Final shape: (1300, 59)


## Drop empty transform columns, drop empty lt4_use, save key completeness summary

In [ ]:

def is_all_missing(series: pd.Series) -> bool:
    return series.isna().all()

# Drop empty derived transform columns (e.g., log1p_ins0 if source variable is fully missing)
transform_prefixes = ("log1p_", "win_")
candidate_transform_cols = [c for c in df_qc.columns if c.startswith(transform_prefixes)]

drop_transform_cols = [c for c in candidate_transform_cols if is_all_missing(df_qc[c])]

if drop_transform_cols:
    df_qc = df_qc.drop(columns=drop_transform_cols)
    print(f"Dropped empty transform columns: {drop_transform_cols}")
else:
    print("No empty transform columns to drop.")

# Drop lt4_use if present but fully missing
if "lt4_use" in df_qc.columns and is_all_missing(df_qc["lt4_use"]):
    df_qc = df_qc.drop(columns=["lt4_use"])
    print("Dropped empty column: lt4_use")
else:
    print("Column lt4_use retained (not present or not fully missing).")

# Key variable completeness summary
key_vars = ["age", "anti_tpo", "tsh", "tg", "hdl", "tc", "glu120"]

rows = []
n_total = len(df_qc)

for col in key_vars:
    if col in df_qc.columns:
        n_non_missing = int(df_qc[col].notna().sum())
        n_missing = int(df_qc[col].isna().sum())
        pct_non_missing = round((n_non_missing / n_total) * 100, 2) if n_total else np.nan
        pct_missing = round((n_missing / n_total) * 100, 2) if n_total else np.nan
        dtype = str(df_qc[col].dtype)
        present_in_dataset = True
    else:
        n_non_missing = 0
        n_missing = n_total
        pct_non_missing = 0.0 if n_total else np.nan
        pct_missing = 100.0 if n_total else np.nan
        dtype = "NOT_PRESENT"
        present_in_dataset = False

    rows.append({
        "variable": col,
        "present_in_dataset": present_in_dataset,
        "n_total": int(n_total),
        "n_non_missing": n_non_missing,
        "pct_non_missing": pct_non_missing,
        "n_missing": n_missing,
        "pct_missing": pct_missing,
        "dtype": dtype,
    })

key_completeness = pd.DataFrame(rows)
key_completeness_path = qc_dir / "qc_key_variable_completeness.csv"
key_completeness.to_csv(key_completeness_path, index=False)

print("Saved:", key_completeness_path)
key_completeness

Dropped empty transform columns: ['log1p_ins0', 'log1p_ins120']
Dropped empty column: lt4_use
Saved: /content/outputs/qc/qc_key_variable_completeness.csv


,variable,present_in_dataset,n_total,n_non_missing,pct_non_missing,n_missing,pct_missing,dtype
0,age,True,1300,1300,100.00,0,0.00,float64
1,anti_tpo,True,1300,1055,81.15,245,18.85,float64
2,tsh,True,1300,1182,90.92,118,9.08,float64
3,tg,True,1300,1183,91.00,117,9.00,float64
4,hdl,True,1300,1183,91.00,117,9.00,float64
5,tc,True,1300,1183,91.00,117,9.00,float64
6,glu120,True,1300,1162,89.38,138,10.62,float64


## Checklist

## S2 completion checklist

- [ ] Review `qc_missingness_report.csv` for variables with extreme missingness.
- [ ] Review `qc_outlier_summary.csv` for sanity-range or unit issues.
- [ ] Review `qc_cohort_availability_summary.csv` for flowchart/STROBE counts.
- [ ] Review `qc_age_summary.csv` for Table 1 and limitations wording.
- [ ] Confirm `tg_hdl_ratio` and `non_hdl` distributions are clinically plausible.
- [ ] Proceed to S3 for explicit cohort/exposure/endpoint definitions.